# Main Crawler

In [ ]:
# Install and import libraries
!pip install beautifulsoup4 requests pandas
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime
import re

In [ ]:
# Filter values for URL construction
JOB_TYPES = [
    'sepenuh-masa',
    'part-time',
    'wfh',
    'hybrid',
    'internship',
    'contract',
    'volunteer'
]

LOCATIONS = [
    # Kuala Lumpur & Putrajaya
    'kuala-lumpur',
    'putrajaya',

    # Selangor
    'selangor',
    'shah-alam',
    'petaling-jaya',
    'subang-jaya',
    'klang',
    'ampang',
    'sepang',
    'hulu-langat',
    'gombak',
    'kuala-selangor',
    'sabak-bernam',
    'kuala-langat',

    # Johor
    'johor',
    'johor-bahru', 
    'iskandar-puteri',
    'batu-pahat',
    'muar',
    'kluang',
    'kota-tinggi',
    'segamat',
    'mersing',
    'pontian',
    'kulai',
    'tangkak',

    # Pulau Pinang
    'pulau-pinang',
    'george-town',
    'butterworth',
    'seberang-perai',
    'batu-ferringhi',
    'bukit-mertajam',
    'nibong-tebal',

    # Perak
    'perak',
    'ipoh',
    'taiping',
    'teluk-intan',
    'manjung',
    'kuala-kangsar',
    'hilir-perak',
    'hulu-perak',
    'kerian',
    'larut-matang',
    'batang-padang',
    'kampar',
    'bagan-datuk',

    # Kedah
    'kedah',
    'alor-setar',
    'sungai-petani',
    'kulim',
    'langkawi',
    'kubang-pasu',
    'kota-setar',
    'baling',
    'padang-terap',
    'yan',
    'pendang',
    'kuala-muda',

    # Kelantan
    'kelantan',
    'kota-bharu',
    'pasir-mas',
    'tumpat',
    'machang',
    'tanah-merah',
    'kuala-krai',
    'gua-musang',
    'bachok',
    'pasir-puteh',
    'jeli',
    'ulu-kelantan',

    # Terengganu
    'terengganu',
    'kuala-terengganu',
    'kemaman',
    'dungun',
    'besut',
    'marang',
    'hulu-terengganu',
    'setiu',

    # Pahang
    'pahang',
    'kuantan',
    'temerloh',
    'bentong',
    'raub',
    'jerantut',
    'cameron-highlands',
    'lipis',
    'rompin',
    'pekan',
    'maran',
    'bera',

    # Negeri Sembilan
    'negeri-sembilan',
    'seremban',
    'nilai',
    'port-dickson',
    'rembau',
    'tampin',
    'jelebu',
    'jempol',
    'kuala-pilah',

    # Melaka
    'melaka',
    'melaka-tengah',
    'alor-gajah',
    'jasin',

    # Sabah
    'sabah',
    'kota-kinabalu',
    'sandakan',
    'tawau',
    'lahad-datu',
    'keningau',
    'semporna',
    'kudat',
    'beaufort',
    'ranau',
    'kota-belud',
    'papar',
    'penampang',
    'putatan',
    'tuaran',

    # Sarawak
    'sarawak',
    'kuching',
    'miri',
    'sibu',
    'bintulu',
    'limbang',
    'sarikei',
    'sri-aman',
    'kapit',
    'mukah',
    'betong',
    'serian',
    'kota-samarahan',
    'lundu',
    'samarahan',

    # Perlis
    'perlis',
    'kangar',
    'arau',
    'padang-besar',

    # Labuan
    'labuan',
]

# Mapping for jtype parameter
JTYPE_MAPPING = {
    'sepenuh-masa': 'jtype=1-Full-Time',
    'part-time': 'jtype=1-Part-Time',
    'internship': 'jtype=1-Internship',
    'contract': 'jtype=1-Contract',
    'volunteer': 'jtype=1-Volunteer',
    'wfh': 'wfh=true',
    'hybrid': 'hybrid=true'
}

# Request headers to mimic browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}
MAX_PAGES = 167        # The website only shows 167 pages for each search

In [ ]:
# Scraping Functions
def get_soup(url, max_retries=3):
    """Fetch page and return BeautifulSoup object with retry logic"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.RequestException as e:
            print(f"  Attempt {attempt + 1} failed for {url}: {e}")
            if attempt < max_retries - 1:
                time.sleep(random.uniform(2, 5))
            else:
                return None

def parse_job_listing(job_card):
    """Extract job details from a single job card"""
    job = {}

    try:
        # Job ID and URL - from the parent div's id attribute or the href
        link_elem = job_card.find('a', href=re.compile(r'/job/\d+'))
        if link_elem:
            href = link_elem.get('href', '')
            job['url'] = f"https://www.maukerja.my{href}" if href.startswith('/') else href
            job_id_match = re.search(r'/job/(\d+)', href)
            job['job_id'] = job_id_match.group(1) if job_id_match else href
        else:
            fallback = job_card.find('a', href=True)
            if fallback:
                href = fallback.get('href', '')
                job['url'] = f"https://www.maukerja.my{href}" if href.startswith('/') else href
            else:
                job['url'] = 'N/A'
            job['job_id'] = 'N/A'

        # Job title - inside the font-bold text-truncate-2-line div
        title_div = job_card.find('div', class_=re.compile(r'text-truncate-2-line'))
        if title_div:
            title_a = title_div.find('a')
            job['title'] = title_a.get_text(strip=True) if title_a else 'N/A'
        else:
            job['title'] = 'N/A'

        # Company name - inside the h2 with id starting with "companyName"
        company_h2 = job_card.find('h2', id=re.compile(r'^companyName'))
        job['company'] = company_h2.get_text(strip=True) if company_h2 else 'N/A'

        # Salary - inside span with class has-text-primary text-base has-text-weight-semibold
        salary_span = job_card.find('span', class_=re.compile(r'has-text-primary.*has-text-weight-semibold|has-text-weight-semibold.*has-text-primary'))
        if salary_span:
            # Strip the SVG icon text, get only the text nodes
            salary_text = salary_span.get_text(strip=True)
            # Remove the dollar sign SVG text artifact if present
            job['salary'] = re.sub(r'^\s*', '', salary_text).strip()
        else:
            job['salary'] = 'N/A'

        # Job type - inside the jobtype-link anchor
        jobtype_a = job_card.find('a', class_='jobtype-link')
        job['job_type'] = jobtype_a.get_text(strip=True) if jobtype_a else 'N/A'

        # Location - inside joblocation-link anchors
        location_parts = job_card.find_all('a', class_='joblocation-link')
        if location_parts:
            job['location'] = ', '.join(a.get_text(strip=True) for a in location_parts)
        else:
            # Fallback: look for the location paragraph
            loc_p = job_card.find('p', class_=re.compile(r'whitespace-nowrap'))
            job['location'] = loc_p.get_text(strip=True) if loc_p else 'N/A'

        # Date posted - inside the italic div at the bottom
        date_div = job_card.find('div', class_=re.compile(r'is-italic'))
        if date_div:
            date_span = date_div.find('span')
            job['date_posted'] = date_span.get_text(strip=True) if date_span else 'N/A'
        else:
            job['date_posted'] = 'N/A'

        # Skills - collect all skill tag links
        skills_div = job_card.find('div', id='jobCardSkills')
        if skills_div:
            skill_links = skills_div.find_all('a', class_=re.compile(r'rounded-full'))
            job['skills'] = ', '.join(a.get_text(strip=True) for a in skill_links)
        else:
            job['skills'] = 'N/A'

    except Exception as e:
        print(f"  Error parsing job card: {e}")
        return None

    return job


def scrape_page(url):
    soup = get_soup(url)
    if not soup:
        return []

    job_cards = soup.find_all('div', class_='job-card-lite')

    jobs = []
    for card in job_cards:
        job = parse_job_listing(card)
        
        if job:
            jobs.append(job)

    return jobs

In [ ]:
def scrape_combination(job_type, location=None, max_pages=MAX_PAGES):
    """Scrape all pages for a job type + location combination"""
    all_jobs = []

    if job_type:
        if location:
            jtype = JTYPE_MAPPING.get(job_type, 'jtype=1-Full-Time')
            base = f"https://www.maukerja.my/jobsearch/jobs-in-{location}?{jtype}"
            urls_generator = lambda p: base if p == 1 else f"{base}&page={p}"
            combo_name = f"{job_type} in {location}"
        else:
            base = f"https://www.maukerja.my/jobsearch/{job_type}-jobs"
            urls_generator = lambda p: base if p == 1 else f"{base}?page={p}"
            combo_name = f"{job_type} (all locations)"
    else:
        if location:
            base = f"https://www.maukerja.my/jobsearch/jobs-in-{location}"
            urls_generator = lambda p: base if p == 1 else f"{base}&page={p}"
            combo_name = f"Jobs in {location}"
        else:
            base = "https://www.maukerja.my/jobsearch/all-jobs?sortBy=date&created=now-1d"
            urls_generator = lambda p: base if p == 1 else f"{base}&page={p}"
            combo_name = f"Most recent jobs"

    print(f"\nScraping: {combo_name}")
    print(f"Start combo scrap time: {datetime.now()}")

    consecutive_empty = 0

    for page in range(1, max_pages + 1):
        url = urls_generator(page)
        jobs = scrape_page(url)
        job_count = len(jobs)

        if not jobs:
            consecutive_empty += 1
            if consecutive_empty >= 3:
                print(f"  No more jobs found after page {page - 2}. Moving to next combination.")
                break
        else:
            consecutive_empty = 0
            all_jobs.extend(jobs)

            if job_count < 27:  # Should have 30 jobs per page
                print(f"  Significant less jobs on page {page} ({job_count} jobs/page). Moving to next combination.")
                break

        if page % 10 == 0:
            print(f"  Page {page}: {len(all_jobs)} jobs collected so far")

        time.sleep(random.uniform(1, 3))

    print(f"  Completed: {len(all_jobs)} jobs from {combo_name}")
    print(f"End combo scrap time: {datetime.now()}")
    return all_jobs

In [ ]:
# Different type for different filters
def run_full_scrape_job_types(seen_job_ids):
    print("\n--- Job Types Only ---")
    for job_type in JOB_TYPES:
        jobs = scrape_combination(job_type, location=None)
        new_jobs = []
        for job in jobs:
            if job['job_id'] not in seen_job_ids:
                seen_job_ids.add(job['job_id'])
                job['source_combo'] = f"{job_type}-all"
                new_jobs.append(job)
        if new_jobs:
            filename = f"../data/raw_data/maukerja_{job_type}_all_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
            pd.DataFrame(new_jobs).to_csv(filename, index=False, encoding='utf-8-sig')
            # files.download(filename)
            print(f"  ✓ Saved {filename} ({len(new_jobs)} jobs)")

def run_full_scrape_location(seen_job_ids):
    print("\n--- Location Only ---")
    for location in LOCATIONS:
        jobs = scrape_combination(None, location=location)
        new_jobs = []
        for job in jobs:
            # if job['job_id'] not in seen_job_ids:
            #     seen_job_ids.add(job['job_id'])
                job['source_combo'] = f"{location}-all"
                new_jobs.append(job)
        if new_jobs:
            filename = f"../data/raw_data/maukerja_{location}_all_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
            pd.DataFrame(new_jobs).to_csv(filename, index=False, encoding='utf-8-sig')
            # files.download(filename)
            print(f"  ✓ Saved {filename} ({len(new_jobs)} jobs)")

def run_full_scrape_job_types_location(seen_job_ids):
    print("\n--- Location + Job Type Combinations ---")
    for location in LOCATIONS:
        for job_type in JOB_TYPES:
            jobs = scrape_combination(job_type, location=location)
            new_jobs = []
            for job in jobs:
                # if job['job_id'] not in seen_job_ids:
                #     seen_job_ids.add(job['job_id'])
                    job['source_combo'] = f"{job_type}-{location}"
                    new_jobs.append(job)
            if new_jobs:
                filename = f"../data/raw_data/maukerja_{job_type}_{location}_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
                pd.DataFrame(new_jobs).to_csv(filename, index=False, encoding='utf-8-sig')
                # files.download(filename)
                print(f"  ✓ Saved {filename} ({len(new_jobs)} jobs)")

def run_full_scrape_recent_jobs(seen_job_ids):
    jobs = scrape_combination(None, None)
    new_jobs = []
    for job in jobs:
        # if job['job_id'] not in seen_job_ids:
        #     seen_job_ids.add(job['job_id'])
            job['source_combo'] = f"recent_jobs{datetime.now()}"
            new_jobs.append(job)
    if new_jobs:
        filename = f"../data/raw_data/maukerja_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
        pd.DataFrame(new_jobs).to_csv(filename, index=False, encoding='utf-8-sig')
        # files.download(filename)
        print(f"  ✓ Saved {filename} ({len(new_jobs)} jobs)")


In [ ]:
# Run the scraper
seen_job_ids = set()        # Not in use anymore due to the difference in url format that causes the failure of the regex function

print(f"Start time: {datetime.now()}")

# run_full_scrape_job_types(seen_job_ids)
# run_full_scrape_location(seen_job_ids)
# run_full_scrape_job_types_location(seen_job_ids)
run_full_scrape_recent_jobs(seen_job_ids)       # run for several days for new data

print(f"End time: {datetime.now()}")